[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_C.ipynb)

# Set C — Cellular Models & Electrophysiology Basics
**Author: Neural Engineering Laboratory, University of Missouri-Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair**

## Introduction
Welcome to the Cellular Modeling module. This lab is designed to move you from a qualitative understanding of neurons to a **quantitative appreciation** of how they function. We achieve this by exploring the spectrum of neural abstraction: from the computationally efficient **LIF** and **Izhikevich** models used in large-scale networks, to the biophysically detailed **Hodgkin-Huxley (HH)** equations that simulate actual ion channel kinetics.

By the end of this module, you will be able to map biological phenomena (like refractory periods and thresholding) to specific mathematical parameters and observe how measured physical signals can be used to drive simulated neural activity.

> ### NOTE for Learners
> While the notebook begins with simplified models (C1-C2), these abstractions assume a firm grasp of electrophysiology principles. **It is strongly recommended that you complete the [B-Appendix: Electrophysiology Basics](#B-Appendix) PRIOR to beginning the contents in Set C.** The Appendix acts as your "Experimental Bench," where you will learn how to measure input resistance ($R_{in}$) and time constants ($\tau$) — concepts essential for interpreting the models in the main set.

## Table of Contents
* **[Introduction](#Introduction)** — Module overview and goals.
* **[C0 Starter](#C0-Starter)** — Environment setup (Run this first!).
* **[B-Appendix: Electrophysiology Basics](#B-Appendix)** — **Start Here** to learn the "Experimental Mindset."
* **[C1 Leaky Integrate-and-Fire (LIF)](#C1-LIF)** — The fundamental RC integrator.
* **[C2 Izhikevich Model](#C2-Izhikevich)** — Adding adaptation and bursting with low cost.
* **[C3 Hodgkin-Huxley via NEURON](#C3-HH)** — Detailed channel kinetics and biophysics.
* **[C4 Real-world Data Bridge](#C4-microbit)** — Mapping sensor data (micro:bit) to current clamp.
* **[Reflection & Discussion](#Reflection)** — Synthesizing biology-to-model mapping.

## C0 Starter — Setup & Readiness Check
Before we begin, we must initialize the programming environment. This cell installs the **NEURON** simulator and ensures the necessary libraries are ready for the Appendix and C3.

### Interactive Module Note
This notebook is designed as an **Interactive Experimental Rig**.
* **Sliders:** Use the sliders on the right of code cells to adjust parameters.
* **Auto-Run:** Most cells are set to `run: "auto"`, meaning the plots will update the moment you move a slider.

In [ ]:
# Install dependencies with quotes around the version constraint
!pip install -q neuron "numpy<2.1" matplotlib --upgrade

import neuron
from neuron import h
import numpy as np
import matplotlib.pyplot as plt

# Ensure standard run functions are available
h.load_file('stdrun.hoc')

print("NEURON is ready and functional.")


## C1 — Leaky Integrate-and-Fire (LIF)
### Interactive Exercise: The Temporal Integrator
The LIF model is a "leaky" bucket for charge. Use the sliders to find the balance between how fast the cell leaks ($tau$) and how much it resists change ($R$).

**Task:** Adjust the **tau** slider to 0.040 (40ms). Does the spike count increase or decrease compared to the baseline? Why?

---
### Exercises: The Temporal Integrator
**Exercise 1: Time Constant Impact**
1. **Predict:** If you double the time constant ($\tau$) from 20ms to 40ms, will the neuron require a *higher* or *lower* frequency of input pulses to reach the threshold?
2. **Verify:** Change `tau = 40e-3` in the code cell above. Observe the "Spikes" count in the output. Did it change as you expected?

**Exercise 2: Resistance vs. Threshold**
1. **Predict:** If you increase the Input Resistance ($R$) from 100 to 150, will the "Spikes" count increase or decrease?
2. **Verify:** Adjust $R$ to 150 and run the cell. How does the voltage slope change during the stimulus?
---

In [ ]:
#@title LIF Experimental Rig { run: "auto" }
import numpy as np
import matplotlib.pyplot as plt

# Parameters from sliders
R = 100 #@param {type:"slider", min:50, max:250, step:5}
tau = 0.02 #@param {type:"slider", min:0.005, max:0.1, step:0.001}
V_rest, V_th, V_reset = -65, -50, -70

t = np.arange(0, 1.0, 1e-4)
I = np.zeros_like(t)
I[(t>0.1)&(t<0.6)] = 0.5; I[(t>0.7)&(t<0.8)] = 0.8
V = np.ones_like(t)*V_rest; spikes = []

for k in range(1, len(t)):
    V[k] = V[k-1] + ((-(V[k-1]-V_rest) + R*I[k-1])*(1e-4/tau))
    if V[k] >= V_th:
        spikes.append(t[k]); V[k] = V[k-1] = V_reset

plt.figure(figsize=(8, 3.5))
plt.plot(t, V, 'k'); plt.ylabel('V (mV)')
plt.twinx(); plt.plot(t, I, 'C0', alpha=0.3); plt.ylabel('I (nA)')
plt.title(f'LIF Response | Spikes: {len(spikes)}')
plt.show()

## C2 — Izhikevich Model
### Exercise: Tuning Firing Patterns
This model uses four parameters ($a, b, c, d$) to replicate complex biology.
* **Regular Spiking (RS):** Use the default slider values.
* **Bursting:** Set **c** to -50 and **d** to 2.

---
### Exercises: Adaptation & Bursting
**Exercise 1: Spike Frequency Adaptation**
1. **Predict:** Using the **Regular Spiking (RS)** parameters, will the interval between the 1st and 2nd spike be the same as the interval between the 5th and 6th?
2. **Verify:** Run the simulation. Use the plot to measure the Inter-Spike Interval (ISI) between early spikes versus late spikes.

**Exercise 2: Parameter Sensitivity**
1. **Predict:** The variable $d$ represents the "reset" of the recovery variable $u$. If you decrease $d$ from 8 to 2, will the neuron's firing rate increase or decrease?
2. **Verify:** Change $d=2$ in the RS parameters and run the cell.
3. **Challenge:** Change the parameters to $a=0.02, b=0.2, c=-50, d=2$ (**Bursting**). Identify the time length of the quiet period between bursts.
---

In [ ]:
#@title Izhikevich Parameter Tuning { run: "auto" }
import numpy as np
import matplotlib.pyplot as plt

a = 0.02 #@param {type:"slider", min:0.01, max:0.1, step:0.01}
b = 0.2 #@param {type:"slider", min:0.05, max:0.3, step:0.01}
c = -65 #@param {type:"slider", min:-75, max:-45, step:1}
d = 8 #@param {type:"slider", min:0.05, max:10.0, step:0.05}

T = 1000; I = np.zeros(T); I[200:800] = 10
v = -65 * np.ones(T); u = b * v

for k in range(1, T):
    dv = 0.04*v[k-1]**2 + 5*v[k-1] + 140 - u[k-1] + I[k-1]
    du = a*(b*v[k-1] - u[k-1])
    v[k] = v[k-1] + dv; u[k] = u[k-1] + du
    if v[k] >= 30:
        v[k-1]=30; v[k]=c; u[k]+=d

plt.figure(figsize=(8, 3.5)); plt.plot(v, 'k')
plt.title('C2: Interactive Izhikevich'); plt.ylabel('v (mV)'); plt.show()

## C3 — HH via NEURON

### Model Idea

Biophysical $Na^+$ / $K^+$ conductances produce threshold and refractoriness from channel kinetics; waveforms reflect identifiable mechanisms.

### Interactive Exercise: Biophysical Kinetics
Use the sliders in the rig below to explore how ionic conductances dictate spike timing and recovery.

**Exercise 1: Kinetic Latency & Timing**
* **The Goal:** Observe how stimulus strength affects the "time-to-fire."
* **Task:** Move the `stim_delay` to 100ms. Now, slowly increase the `stim_amp` from 0.08 to 0.15 nA.
* **Observe:** Does the first spike move closer to the dotted blue "Onset" line? This represents the time required for sodium channels to reach the open-state threshold.

**Exercise 2: The Refractory Barrier & AHP**
* **The Goal:** Understand Afterhyperpolarization (AHP).
* **Observe:** Look at the "dip" in voltage immediately following a spike.
* **Task:** Adjust `stim_amp` to a high value (e.g., 0.25 nA) to cause rapid firing.
* **Verify:** Note how the cell cannot fire again until the voltage recovers from that dip. This is where $K^+$ gates are still closing, creating the refractory period you measured in the Appendix.

In [ ]:
#@title C3: Hodgkin-Huxley Experimental Rig { run: "auto" }
from neuron import h
import numpy as np
import matplotlib.pyplot as plt

# 1. Clear memory first
h('forall delete_section()')

# 2. Parameters from Sliders
stim_amp = 0.1 #@param {type:"slider", min:0.05, max:0.3, step:0.01}
stim_dur = 600 #@param {type:"slider", min:100, max:800, step:50}
stim_delay = 100 #@param {type:"slider", min:10, max:300, step:10}

# 3. Rig Setup
soma = h.Section(name='soma_hh')
soma.L = soma.diam = 12.6157
soma.insert('hh')

stim = h.IClamp(soma(0.5))
stim.delay = stim_delay
stim.dur = stim_dur
stim.amp = stim_amp

# 4. Recording & Run
v = h.Vector().record(soma(0.5)._ref_v)
t = h.Vector().record(h._ref_t)
h.finitialize(-65)
h.continuerun(800)

# 5. Plotting
plt.figure(figsize=(8, 4))
plt.plot(t, v, 'k', lw=1.5, label='Membrane Voltage')
plt.axhline(-65, color='gray', linestyle='--', alpha=0.5)
plt.axvline(stim_delay, color='C0', linestyle=':', alpha=0.8, label='Stimulus Onset')
plt.title(f'HH Current Clamp | Delay: {stim_delay}ms | Amp: {stim_amp}nA')
plt.xlabel('Time (ms)'); plt.ylabel('Membrane Voltage (mV)')
plt.legend(loc='upper right'); plt.grid(alpha=0.2)
plt.show()

## C4 — Real-world Data Bridge

### Model Idea

In a Brain-Computer Interface (BCI) or sensory prosthetic, we don't use sliders—we use **Sensors**. However, sensors often output "Raw Units" (like 0-1023) that don't match the "Biophysical Units" (nA) of our model.

### Interactive Exercise: The Scaling Challenge
**The Goal:** Map raw sensor data to the `stim.amp` of your HH model to trigger meaningful spikes.

1. **Observe the Input:** The top (gray) plot shows a signal recorded from an external device.
2. **Adjust `data_scale`:** This acts as your "Gain." If it's too low, the signal won't reach the **Rheobase** you discovered in the Appendix.
3. **Adjust `data_offset`:** This moves the entire signal up or down. Use this to set the "Baseline" firing rate.

**Exercise: Precision Triggering**
* **Task:** Adjust the sliders so that the neuron fires **exactly one spike** during the highest peak of the sensor data, but remains silent during the smaller ripples.
* **Reflect:** In a prosthetic limb, why is it critical to tune the `data_offset` so the model doesn't "fire" when the sensor is just reading background noise?

In [ ]:
#@title C4: Sensor-to-Soma Data Bridge { run: "auto" }
data_scale = 0.005 #@param {type:"slider", min:0.001, max:0.02, step:0.001}
data_offset = 0.05 #@param {type:"slider", min:-0.1, max:0.2, step:0.01}

from neuron import h
import numpy as np
import matplotlib.pyplot as plt

# 1. Create a dummy "Real-world" sensor signal (e.g., an Accelerometer)
t_data = np.linspace(0, 500, 1000)
raw_sensor = 50 + 30 * np.sin(2 * np.pi * 0.01 * t_data) + 10 * np.random.randn(1000)

# 2. Map Sensor -> Model Amps
# We apply the user's scaling and offset here
mapped_amps = (raw_sensor * data_scale) + data_offset

# 3. Setup NEURON Rig
h('forall delete_section()')
soma_c4 = h.Section(name='somac4')
soma_c4.L = soma_c4.diam = 20
soma_c4.insert('hh')

# Use play() to drive the injection with the mapped array
stim_vec = h.Vector(mapped_amps)
t_vec_stim = h.Vector(t_data)
stim = h.IClamp(soma_c4(0.5))
stim.delay = 0
stim.dur = 1e9 # Keep open to follow the vector
stim_vec.play(stim._ref_amp, t_vec_stim, 1)

# Record
v = h.Vector().record(soma_c4(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

h.finitialize(-65)
h.continuerun(500)

# 4. Plotting
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

ax1.plot(t_data, raw_sensor, color='gray', alpha=0.5)
ax1.set_ylabel('Raw Sensor Units')
ax1.set_title('External Data Input')

ax2.plot(t, v, 'k')
ax2.set_ylabel('Membrane V (mV)')
ax2.set_xlabel('Time (ms)')
ax2.axhline(-50, color='red', ls=':', alpha=0.5, label='Threshold')

plt.tight_layout()
plt.show()

## Practice / Discussion Questions — Set C — Biology → Model Mapping

1. **Model Abstraction:** Define a "model abstraction" for a neuron that preserves interpretability. Which parameters must stay in **biophysical units** (e.g., mV, nA), and why?
2. **The Power of the Clamp:** In **B-A.3**, you used a **Voltage Clamp slider** to "unmask" $Na^+$ and $K^+$ currents. Why is it impossible to see these individual currents clearly in a standard Current Clamp?
3. **Geometric Scaling:** Based on your exploration in **B-A.4**, if you modeled a large Alpha Motor Neuron (large diameter), how would you adjust the `R` and `tau` sliders in the **LIF model** to stay accurate?
4. **Rheobase Discovery:** In **B-A.2**, you found the **Rheobase**. If you increased the "leakiness" (`g_pas`) of that cell, would you expect the Rheobase to increase or decrease?
5. **Efficiency vs. Detail:** *Justify:* What is gained and lost when moving from a detailed **HH-type model** (C3) to a reduced **Izhikevich model** (C2)?
6. **Refractoriness:** Compare the "Refractory" behavior in **C2 (Izhikevich)** vs **B-A.6 (HH)**. Which model felt more "biological" as you moved the ISI slider?
7. **The Data Bridge:** In **C4**, we map sensor values to `stim.amp`. If your sensor data is "noisy," how might that noise interact with the **Rheobase** threshold?
8. **Parameter Transparency:** Why is "parameter transparency" essential for educational modeling? Give one consequence of using "opaque" scaling (e.g., arbitrary units).
9. **I-V Linearity:** In **B-A.5**, is the I-V curve a perfectly straight line? What biological feature causes it to curve at higher voltages?
10. **Time Constant Intuition:** If two neurons have the same **Input Resistance** but different **Capacitances**, which one will respond faster to a sudden slider change in `stim_amp`?
11. **Reset Dynamics:** In the **Izhikevich model**, the `c` slider sets the reset voltage. How does this mimic the **Afterhyperpolarization (AHP)** you observed in the HH model?
12. **The Role of NEURON:** What is the primary advantage of using a simulator like **NEURON** (C3) over writing your own Euler integration (C1) from scratch?
13. **Assumptions:** How do you **document assumptions** so a reader can reproduce and critique your model? (List 3-4 items).
14. **Validation:** How would you **validate** that your interactive rig is working correctly before using it to collect data for a lab report?
15. **Missing Mechanisms:** Give an example of a **missing mechanism** (e.g., omitting Calcium) that could lead to a consistent mismatch between your model and real data.
16. **Bursting Logic:** In **C2**, you tuned the `d` parameter for bursting. Biologically, what might `d` represent in terms of slow ion channel recovery?
17. **Step Tests:** Why do we advocate for a **"membrane step test"** (B-A.4) as the first step in characterizing any new neural model?
18. **Reproducible Workflows:** Outline a minimal **reproducible workflow** (notebook → plots → metrics) for moving from an Appendix measurement to a Main Set simulation.
19. **Educational Trade-offs:** What is the trade-off of adding **noise sources** to a beginner-level model? Does it help or hinder the learning of core concepts?
20. **Forward Reflection:** What core principle from **Set B** (Cellular) will be most important when you begin analyzing **spatial effects** (Dendrites/Axons) in Set C?